In [59]:
!pip install -q langchain langchain-openai langchain-tavily python-dotenv

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-5.6-luna",
    use_responses_api=True
)

print("LLM ready")

LLM ready


In [3]:
import json

with open("thebayrestaurant_jed.json", "r", encoding="utf-8") as f:
    evidence = json.load(f)

print("Research Evidence loaded")

Research Evidence loaded


In [4]:
from langchain_tavily import TavilySearch
from langchain_core.tools import tool

web_search = TavilySearch(
    max_results=5,
    topic="general",
    search_depth="advanced"
)

@tool
def search_instagram_benchmark(query: str) -> str:
    """Search the web for current Instagram marketing benchmarks relevant to restaurants."""
    
    results = web_search.invoke({
        "query": query
    })
    
    return str(results)

print("Benchmark tool ready")

Benchmark tool ready


In [5]:
test = search_instagram_benchmark.invoke(
    "2026 Instagram posting frequency engagement benchmark for restaurants"
)

print(test)

{'query': '2026 Instagram posting frequency engagement benchmark for restaurants', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://evokad.com/restaurant-social-media-marketing-guide-2026', 'title': 'The Restaurant Social Media Marketing Guide 2026', 'content': 'For restaurants specifically, the platform picture gets more nuanced. Hootsuite’s industry benchmarks place dining and hospitality brands at around 3.1% engagement on Instagram, well above the cross-industry average. That premium exists because food content performs natively on visual platforms. What most CMOs miss is how much the numbers shift when follower size is factored in. Accounts with fewer than 10,000 followers average 4.7% engagement on TikTok and 2.8% on Instagram, significantly higher than those with more than 100,000 followers. [...] The highest-converting Instagram content mix for restaurants combines Reels for reach, carousels for depth, and Stories for reservation prompts. 

In [7]:
evidence_text = json.dumps(
    evidence,
    indent=2,
    ensure_ascii=False
)

print(evidence_text[:1000])

{
  "restaurant": {
    "restaurant_id": 3,
    "name": "The Bay Restaurant",
    "instagram_username": "thebayrestaurant_jed",
    "instagram_url": "https://www.instagram.com/thebayrestaurant_jed/",
    "email": null,
    "location": "Jedaah",
    "category": "Cafe, Restaurant, Food & Beverage Company"
  },
  "analysis_metadata": {
    "status": "partial",
    "analyzed_at": "2026-09-14T07:41:51.804323+00:00",
    "posts_scraped": 6,
    "posts_analyzed": 5,
    "posts_failed": 1,
    "source": "instagram",
    "analysis_version": "1.0"
  },
  "profile": {
    "username": "thebayrestaurant_jed",
    "full_name": "•The BAY• 🍃 •ذا باي•",
    "bio": "Refined Indian Cuisine 🇮🇳 \nBold Flavors, Modern Soul.\n📍The bay, Jeddah",
    "followers": 20474,
    "following": 1,
    "posts_count": 432,
    "website": "https://linktr.ee/thebayrestaurant_jed?utm_source=linktree_profile_share&ltsid=e5bd1dd7-599f-4005-84d2-9819fd8d0f9e",
    "verified": false,
    "business_category": "Restaurant",
    

In [25]:
qualification_prompt = f""" 
You are the Qualification & Marketing Gap Analysis Agent for Rawaj. 
 
Your task is to analyze ONE restaurant using its Instagram research evidence 
and determine whether the restaurant has meaningful marketing gaps. 
 
The restaurant has already passed the ICP criteria. 
DO NOT re-evaluate the ICP. 
 
Your role is ONLY: 
- Analyze the restaurant's current Instagram marketing performance. 
- Identify marketing gaps supported by evidence. 
- Compare relevant metrics with external benchmarks when useful. 
- Explain why each finding represents a marketing gap. 
- Determine whether the restaurant should be qualified for Rawaj's marketing services. 
 
DO NOT generate a marketing strategy. 
DO NOT recommend solutions. 
DO NOT calculate revenue impact. 
DO NOT invent missing data. 
 
==================== 
RESEARCH EVIDENCE 
==================== 
 
{evidence_text} 
 
==================== 
BENCHMARK SEARCH 
==================== 
 
You have access to the Benchmark Search Tool. 
 
Use the Benchmark Search Tool whenever a marketing metric can meaningfully 
be evaluated against an external industry benchmark. 
 
Prioritize benchmarks that are: 
- Recent, preferably 2026 
- Specific to Instagram 
- Specific to Food & Beverage / Restaurants when available 
- From credible research companies, benchmark reports, or established 
  marketing research sources 
 
Important: 
Do NOT assume that a benchmark is a universal rule. 
A benchmark is a reference point based on research, not a mandatory requirement. 
 
For each benchmark-based finding: 
- State the restaurant's actual metric. 
- State the benchmark metric/value. 
- State the benchmark context (Food & Beverage, Restaurant, Instagram, etc.). 
- State the source/report name. 
- State the publication year. 
- State the source URL when available. 
- Compare the restaurant's result with the benchmark. 
- Explain what the difference means from a marketing perspective. 
 
If multiple credible benchmarks are found and they differ, mention the relevant 
benchmarks and explain the difference instead of selecting a number arbitrarily. 
 
If no reliable benchmark is available, do NOT invent one. 
Use the restaurant's own evidence and clearly state that the gap was identified 
from the restaurant's evidence rather than an external benchmark. 
 
==================== 
IMPORTANT BENCHMARK REASONING 
==================== 
 
When a metric appears weak, do not simply label it as "low" or "poor". 
 
The analysis MUST explain how the gap was identified. 
 
For example: 
 
Instagram Evidence: 
The restaurant's engagement rate is 0.031%. 
 
Benchmark Evidence: 
The Benchmark Search Tool found a 2026 Food & Beverage Instagram engagement 
benchmark of approximately 0.4% from a named benchmark report. 
 
Comparison: 
The restaurant's 0.031% engagement rate is substantially below the 0.4% 
industry reference. 
 
Conclusion: 
This comparison supports identifying Low Instagram Engagement as a 
marketing gap. 
 
Apply the same reasoning to other measurable metrics such as: 
- Engagement rate 
- Posting frequency 
- Content frequency 
- Video/content format usage 
- Other measurable Instagram performance indicators 
 
Do NOT say "restaurants should always post X times per week" unless the source 
actually states that as a benchmark or recommendation. 
Instead, describe it as an industry benchmark, average, reference, or target 
according to the source. 
 
==================== 
ANALYSIS REQUIREMENTS 
==================== 
 
Analyze the restaurant in detail. 
 
1. RESTAURANT OVERVIEW 
 
Extract and explain the important information available about the restaurant, 
including where available: 
 
- Restaurant name 
- Location 
- Instagram username 
- Followers 
- Total posts 
- Recent posting activity 
- Posting frequency 
- Days since last post 
- Average likes 
- Average comments 
- Engagement rate 
- Content formats 
- Content themes 
- CTA usage 
- Promotional content 
- Menu visibility 
- Price visibility 
- Offer visibility 
- Branding consistency 
- Any other relevant Instagram evidence 
 
For important metrics, mention where the information came from 
(e.g. Instagram profile data, Instagram metrics, or content analysis). 
 
Do not mention information that is not available. 
 
2. MARKETING PERFORMANCE ASSESSMENT 
 
Evaluate the overall Instagram marketing performance. 
 
Discuss separately: 
 
- Audience engagement 
- Posting consistency 
- Content quality and variety 
- Product/menu visibility 
- Promotional communication 
- CTA and customer action 
- Conversion-oriented information 
- Brand presentation 
- Audience interaction 
 
For every assessment, clearly distinguish between: 
 
A. Direct Instagram Evidence 
What was actually observed or measured. 
 
B. External Benchmark Evidence 
If a relevant benchmark exists, state what the web research found, 
including the source and year. 
 
C. Comparison 
Explain how the restaurant compares with the benchmark. 
 
D. Marketing Interpretation 
Explain what the comparison means and whether it indicates a weakness. 
 
Do not turn assumptions into evidence. 
 
3. MARKETING GAPS 
 
Identify ALL meaningful marketing gaps supported by the evidence. 
 
For EVERY gap, provide the following: 
 
- Gap name 
- What was observed 
- Exact Instagram evidence 
- Evidence source 
- Relevant metric, if available 
- External benchmark, if relevant 
- Benchmark source/report name 
- Benchmark year 
- Benchmark URL when available 
- Comparison between restaurant and benchmark 
- Explanation of why the comparison indicates a gap 
- Marketing interpretation 
- Affected marketing area 
- Affected customer journey stage 
- Severity: High / Moderate / Low 
- Confidence: High / Moderate / Low 
- Priority: 1 = highest priority 
 
VERY IMPORTANT: 
 
Do not simply write: 
"Posting frequency is low." 
 
Instead explain: 
 
"The Instagram research shows that the restaurant publishes 0.93 posts 
per week. The Benchmark Search Tool found a 2026 Food & Beverage Instagram 
posting benchmark of approximately X posts per week from [SOURCE]. 
The restaurant's publishing frequency is below this benchmark, which 
supports identifying Posting Frequency as a marketing gap." 
 
Do the same for engagement and every other metric where a reliable benchmark 
can be found. 
 
For non-metric gaps, such as poor menu visibility or inconsistent CTA usage, 
use the actual content analysis as evidence and explain why the observed 
pattern represents a marketing weakness. 
 
Every gap MUST have a clear evidence-based explanation of: 
 
WHAT was found → WHERE it came from → WHAT the benchmark/research says 
(if applicable) → HOW the restaurant compares → WHY this is a gap. 
 
Do not create a gap just because something is different. 
Only identify something as a gap when there is reasonable evidence 
that it represents a marketing weakness. 
 
4. STRENGTHS 
 
Identify the important marketing strengths shown by the evidence. 
 
For each strength: 
- State the observed evidence. 
- Explain why it is a strength. 
- Do not claim performance benefits that cannot be supported by the data. 
 
5. DATA LIMITATIONS 
 
Clearly mention limitations in the research. 
 
For example: 
- incomplete post analysis 
- unavailable reach 
- unavailable impressions 
- unavailable saves 
- unavailable shares 
- unavailable profile visits 
- unavailable conversions 
- missing posts 
- failed analysis 
- limited sample size 
- differences between benchmark methodologies 
 
Do not treat missing data as evidence of poor performance. 
 
6. FINAL DECISION 
 
At the end, clearly answer: 
 
Does this restaurant have meaningful marketing gaps? 
 
Answer: 
YES or NO 
 
Then determine: 
 
Qualification: 
Qualified or Unqualified 
 
Explain the decision based only on the available evidence. 
 
The final decision should reference the most important identified gaps 
and their severity/priority. 
 
==================== 
IMPORTANT RULES 
==================== 
 
- Use only available restaurant evidence. 
- Do not invent engagement, reach, conversion, revenue, customer, or sales data. 
- Do not assume missing information is zero. 
- Do not confuse absence of evidence with evidence of absence. 
- Do not generate solutions or strategies. 
- Do not calculate revenue impact. 
- Be specific and detailed. 
- Prioritize evidence over assumptions. 
- Use the Benchmark Search Tool when a relevant benchmark can strengthen 
  the evaluation. 
- Prefer recent 2026 benchmarks. 
- Always name the source when using an external benchmark. 
- Never present a benchmark as a universal rule. 
- If different credible sources report different benchmarks, report the 
  difference clearly. 
- Never invent a source, benchmark value, study, or URL. 
- If evidence is insufficient, explicitly say so. 
- Every identified gap must have an explanation showing how the gap was 
  determined. 
 
==================== 
OUTPUT FORMAT 
==================== 
 
Return the analysis as a detailed plain-text report. 
 
Do NOT return JSON. 
Do NOT return Python objects. 
Do NOT return dictionaries. 
Do NOT return code. 
 
Use clear headings and bullet points. 
 
For every marketing gap, use this structure: 
 
## Gap: [Gap Name] 
 
Severity: High / Moderate / Low 
Priority: [number] 
Confidence: High / Moderate / Low 
 
### Instagram Evidence 
[What the restaurant research found] 
 
### Evidence Source 
[Where the evidence came from] 
 
### External Benchmark Evidence 
[Benchmark value, study/report, year, and source URL if available] 
 
### Comparison 
[Restaurant metric vs benchmark] 
 
### Why This Is a Marketing Gap 
[Detailed explanation connecting the evidence and comparison] 
 
### Marketing Area 
[Area affected] 
 
### Customer Journey Stage 
[Stage affected] 
 
If no external benchmark was used, write: 
"No external benchmark used — gap identified from Instagram research evidence." 
 
The final report should be detailed enough for Rawaj's next agent 
to understand exactly which gaps exist, why they were identified, 
how strong the evidence is, and which gaps should be addressed first.
"""

In [26]:
llm_with_tools = llm.bind_tools([
    search_instagram_benchmark
])

print("Tool connected")

Tool connected


In [27]:
response = llm_with_tools.invoke(qualification_prompt)

print(response)

content=[{'id': 'rs_05719bb7f0febc47006aa985a42dec87d2a43dc2f27f34dc98', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqqYWlSEnH2QrkS8R0I2_RrsKJGr9iwno6HkH07URnwSTus-Eq8LfTpAekiEqkmLU9xAs8mHkQwPt6y5m3R_Ftx8gU9q6-K7DO897TXUsibIrqqA9g61qe1eTznKeZxYtcybD2Nt0Y2jvp8PsTcaFzmImRZpG09ATSDNM4Ty_XULjorcIsQdjdiKtstwdh-lnU1gD0_AZzdppsXDsQjbKyGkGh-UcUfW0oF4LjB9w0Ud3srM_Ioih0l0LjVdd-iyNAkntrmN3QLX1I4YAEkvekrYTKcNmhnaC9L9m3qhdudcgAGGXE_n4K4leDLBI1wyaLKoCKQJQVFBFbhHvoyVhcZWRbVy88TUeP1t3NSYcExApTJ-bhGjqn-KYAi7dHG8x08ZdFSp3Cof69Cau-kZ0WrSn3BhYfMa4XMZ8UtD3gGCJ1L0-3OPU5zalZabHlLVcxkHRmNjtbZnTLK_yf9l5UnNXpNVNed4Bdtgs0amtkLK6tEwCUgIo5XCXMYxHpRzM7eA8lVa5_K8IVThQGlRxLBClltOdoh9qeTzrA9UZ4_bBZhoMrJuGummobIQe4MHGpCcH822n5bdY9icGdN8HTQOyVl3PRN70b9LjJkcnuz4MhDpRFN26hDpejYPFc-_kZMwlNRf1dYOltT92Tq5rHCYNI14Dd5sRa1s1-mw4cdwonHe9faGpDFZ9f3hY_6OOVzDzXuMZ2TjC6i-dMqcOW4nnMziLDsCjAZ8-rZVNY_9aj7Dn50Nat2i3zgdm-qDt6hYTO23fe4A0t-0gHPbPrF9r07-Paeg_YM-tgEo-qIU61Aisl4dZI9Lm5tN2YPZxGMkCYPdtYHW5c9m

In [28]:
tool_result = search_instagram_benchmark.invoke(
    response.tool_calls[0]["args"]
)

print(tool_result)

{'query': '2026 Instagram engagement rate benchmark food beverage restaurants report', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://creatorflow.so/blog/instagram-engagement-rate-benchmarks-2026', 'title': 'Instagram Engagement Rate Benchmarks: 15-Niche Table (2026)', 'content': '| Food & Beverage | 1.55% | At average | Reels (recipe videos), Carousels (step-by-step) |\n| Lifestyle | 1.53% | At average | Carousels (routines), Reels (day-in-life) |\n| Education | 1.48% | -5% below | Carousels (infographics), Reels (explainers) |\n| Finance & Business | 1.34% | -14% below | Carousels (tips), Static (quotes) |\n| Technology | 1.31% | -15% below | Reels (demos), Carousels (comparisons) |\n| Real Estate | 1.25% | -19% below | Reels (walkthroughs), Carousels (listings) |\n| Fashion & Apparel | 1.24% | -20% below | Reels (outfit transitions), Carousels (lookbooks) |\n| Makeup & Beauty | 1.19% | -23% below | Reels (tutorials), Carousels (before/after) 

In [30]:
final_prompt = f"""
{qualification_prompt}

====================
BENCHMARK SEARCH RESULTS
====================

{tool_result}

Use these benchmark results in your analysis.

For every benchmark-based gap:
- State the restaurant's actual metric.
- State the benchmark value.
- Mention the study/report name.
- Mention the year.
- Mention the source URL when available.
- Compare the restaurant's metric with the benchmark.
- Explain why the comparison supports identifying the marketing gap.

Now produce the final detailed plain-text Marketing Gap Analysis.
"""

final_response = llm.invoke(final_prompt)

print(final_response.content)

[{'id': 'rs_0f8c6c860ae6c332006aa9881e271887d2b1ec69ffa8e4eeff', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqqYhd-mmbHH9x35bwSdF5wGiEDwH53yZfGyJI6ys1TRdG6njRmb8SJStV1zl0qSz4IIQDICOvXoCEKExYXjCFVCBi4Epi8axIIBjdwhcDIxGzMORG-SXHS5RJzDnPbT8O7_sR-IGnBV1VWD_Z0JFH_qlXT6SkZv6P8tkYolzNbHbCURpCcn6UjqleCUWPUSG4AVfDa65mg6IWZW_Pg5bbJABKdqtjUWTQuTMwLoBv_BAVqEYiLM0b7WKj5lEQ4UoCYBUG48UfORiAYeTeNaIzVrD1RsWlGYxdVDDZz9TDG4rjU2PEKBlsJEjmZfFKoR67lBtU_cafyuxvGFIR3UrWcknglO-ixIG8p37wp2t904q_NvuKwVo8XCO9foWtiL1YCX_ngV1GzHR_X6l1EabBzEYHa_pxTTpl5jLnhvNQbh-B5Lzj1XoWc_s52NsUa842zZhep5tmalOBVhglhppUcJeJRGarJg5cK0ZfQzlhoHlWAHE8MYwtpJvSpYM5zoiD1Ey9m4nRYAgilaJjhvvl0rdz6ZxGrfIOfYMRuNo6OYPhiHc1ebKMWnG6IlSmF8sK0pI6gPvXqyoZaeA94v2r_MEFvqokZCSb9rouSsiBAu-Oj4vB__kPfangaCoKvCqbFMKs1PkTazZ2yHGUU8_l2ZeZ7fBj_JLKxTYR2HJGQAGKh__7BqMzqTyPvqV0BPYQe5JnygDk_pdM6BsrIG523pszhZ-70gsU_VcZ-YzkBJ69MCXJL3fJ8uDoWqO_nlI2bFz3561iUzdpLDuRqt0jKG7PLiSCRbYj_XOZT71Sw02RXBwEtfbK9N1d3nW8Lg7pSqgUAuDlHPg6ol0OJtAjRY